In [1]:
#STEP 0Setup & load data (run once)

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Load processed EPC data
df = pd.read_csv(
    "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_processed.csv"
)

# Define outcome variable

df['COST_INTENSITY_CURRENT'] = df['3_YR_ENERGY_COST_CURRENT'] / 3 / df['TOTAL_FLOOR_AREA']

target = "COST_INTENSITY_CURRENT"

# Drop rows with missing PEI
df = df.dropna(subset=[target])

df.shape

(136, 119)

In [2]:
# ==========================
# Continuous variables correlation with ENERGY_CONSUMPTION_CURRENT
# ==========================

from scipy.stats import pearsonr, spearmanr

# List of continuous variables to test
continuous_vars = [
    "TOTAL_FLOOR_AREA",
    "3_YR_ENERGY_COST_CURRENT",
    "ENERGY_CONSUMPTION_CURRENT",
    "3_YR_ENERGY_SAVINGS_POTENTIAL",
    "CO2_EMISS_CURR_PER_FLOOR_AREA",
    "SPACE_HEATING_DEMAND",
    "WATER_HEATING_DEMAND",
    "GLAZED_AREA",
    "NUMBER_HABITABLE_ROOMS",
    "NUMBER_HEATED_ROOMS",
    "HEATING_COST_CURRENT",
    "HEATING_COST_POTENTIAL",
    "HOT_WATER_COST_CURRENT",
    "HOT_WATER_COST_POTENTIAL",
    "LIGHTING_COST_CURRENT",
    "LIGHTING_COST_POTENTIAL",
    "EXTENSION_COUNT",
    "LOW_ENERGY_FIXED_LIGHT_COUNT"
]

results_continuous = []

for col in continuous_vars:
    if col in df.columns:
        # Drop missing values for this column
        temp = df[[col, target]].dropna()
        # Skip if column is constant
        if temp[col].nunique() > 1:
            pearson_r, pearson_p = pearsonr(temp[col], temp[target])
            spearman_r, spearman_p = spearmanr(temp[col], temp[target])
            results_continuous.append({
                "Variable": col,
                "Pearson_r": pearson_r,
                "Pearson_p": pearson_p,
                "Spearman_r": spearman_r,
                "Spearman_p": spearman_p
            })

# Convert to DataFrame
df_continuous_corr = pd.DataFrame(results_continuous).sort_values("Spearman_r", key=abs, ascending=False)
df_continuous_corr

,Variable,Pearson_r,Pearson_p,Spearman_r,Spearman_p
2,ENERGY_CONSUMPTION_CURRENT,0.716344,1.083921e-22,0.693634,7.982118e-21
3,3_YR_ENERGY_SAVINGS_POTENTIAL,0.613817,1.931114e-15,0.653275,6.705603e-18
1,3_YR_ENERGY_COST_CURRENT,0.410653,6.848314e-07,0.486338,1.941476e-09
10,HEATING_COST_CURRENT,0.386886,3.276491e-06,0.475137,5.072760e-09
12,HOT_WATER_COST_CURRENT,0.500007,5.732363e-10,0.466277,1.058617e-08
4,CO2_EMISS_CURR_PER_FLOOR_AREA,0.480964,3.091359e-09,0.448895,4.224317e-08
0,TOTAL_FLOOR_AREA,-0.212613,1.295287e-02,-0.331689,7.990230e-05
17,LOW_ENERGY_FIXED_LIGHT_COUNT,-0.242524,4.444150e-03,-0.329437,8.991589e-05
11,HEATING_COST_POTENTIAL,0.181507,3.444908e-02,0.266586,1.705792e-03
6,WATER_HEATING_DEMAND,-0.150434,8.044400e-02,-0.190664,2.618755e-02


In [3]:
# ==========================
# Binary variables correlation with ENERGY_CONSUMPTION_CURRENT
# ==========================

from scipy.stats import pointbiserialr

binary_vars = [
    "MAINS_GAS_FLAG",
    "SOLAR_WATER_HEATING_FLAG",
    "FLAT_TOP_STOREY",
    "PHOTO_SUPPLY"
]

results_binary = []

for col in binary_vars:
    if col in df.columns:
        temp = df[[col, target]].dropna()
        # Try to convert to numeric (0/1)
        try:
            temp[col] = temp[col].astype(int)
        except ValueError:
            # If it can't be converted, skip this column
            print(f"Skipping {col}: not numeric or not binary")
            continue
        
        # Only compute correlation if there is more than one unique value
        if temp[col].nunique() > 1:
            r, p = pointbiserialr(temp[col], temp[target])
            results_binary.append({
                "Variable": col,
                "PointBiserial_r": r,
                "p_value": p
            })

df_binary_corr = pd.DataFrame(results_binary).sort_values("PointBiserial_r", key=abs, ascending=False)
df_binary_corr

Skipping PHOTO_SUPPLY: not numeric or not binary


,Variable,PointBiserial_r,p_value
1,FLAT_TOP_STOREY,-0.245511,0.524296
0,SOLAR_WATER_HEATING_FLAG,-0.083946,0.344234


In [4]:
# ==========================
# Categorical variables correlation with ENERGY_CONSUMPTION_CURRENT
# ==========================

from scipy.stats import f_oneway

categorical_vars = [
    "CONSTRUCTION_AGE_BAND",
    "PROPERTY_TYPE",
    "BUILT_FORM",
    "MAIN_HEATING_CATEGORY",
    "MAIN_FUEL",
    "MAIN_HEATING_CONTROLS",
    "MECHANICAL_VENTILATION",
    "ENERGY_TARIFF",
    "TENURE",
    "TRANSACTION_TYPE",
    "DATA_ZONE",
    "DATA_ZONE_2011",
    "LOCAL_AUTHORITY_LABEL",
    "CONSTITUENCY_LABEL"
]

results_categorical = []

for col in categorical_vars:
    if col in df.columns:
        temp = df[[col, target]].dropna()
        groups = [grp[target].values for name, grp in temp.groupby(col)]
        if len(groups) > 1:
            f_stat, p_val = f_oneway(*groups)
            # Eta-squared as effect size
            ss_between = sum(len(g)*(g.mean() - temp[target].mean())**2 for g in groups)
            eta_squared = ss_between / sum((temp[target] - temp[target].mean())**2)
            results_categorical.append({
                "Variable": col,
                "F_stat": f_stat,
                "p_value": p_val,
                "Eta_squared": eta_squared
            })

df_categorical_corr = pd.DataFrame(results_categorical).sort_values("Eta_squared", ascending=False)
df_categorical_corr

,Variable,F_stat,p_value,Eta_squared
5,MAIN_HEATING_CONTROLS,2.466576,0.000871,0.358345
4,MAIN_FUEL,2.780689,0.002272,0.220444
3,MAIN_HEATING_CATEGORY,5.643063,0.000033,0.214486
9,TRANSACTION_TYPE,2.999363,0.005985,0.140914
7,ENERGY_TARIFF,4.202141,0.007187,0.091612
0,CONSTRUCTION_AGE_BAND,1.184022,0.311613,0.083476
6,MECHANICAL_VENTILATION,4.870777,0.009112,0.069216
8,TENURE,3.210818,0.025186,0.068010
1,PROPERTY_TYPE,1.069032,0.364596,0.023720
2,BUILT_FORM,0.547421,0.701213,0.016440


Here’s the structured correlation report for **current cost intensity** (`3_YR_ENERGY_COST_CURRENT` or similar), formatted like the PEI and carbon reports.

---

## **1. Continuous Variables**

**Key results:**

| Variable                                                         | Pearson_r       | Pearson_p | Spearman_r | Spearman_p | Notes                                                                               |
| ---------------------------------------------------------------- | --------------- | --------- | ---------- | ---------- | ----------------------------------------------------------------------------------- |
| ENERGY_CONSUMPTION_CURRENT                                       | 0.716           | 1e-22     | 0.694      | 8e-21      | Very strong positive correlation; energy use drives cost intensity                  |
| 3_YR_ENERGY_SAVINGS_POTENTIAL                                    | 0.614           | 2e-15     | 0.653      | 7e-18      | Moderate-to-strong correlation; high-cost savings potential homes stand out         |
| HOT_WATER_COST_CURRENT                                           | 0.500           | 6e-10     | 0.466      | 1e-08      | Significant contributor; water heating cost drives household energy expenditure     |
| CO2_EMISS_CURR_PER_FLOOR_AREA                                    | 0.481           | 3e-09     | 0.449      | 4e-08      | Reflects link between carbon and cost; higher emissions correlate with higher costs |
| HEATING_COST_CURRENT                                             | 0.387           | 3e-06     | 0.475      | 5e-09      | Heating cost is a major driver of cost intensity                                    |
| 3_YR_ENERGY_COST_CURRENT                                         | 0.411           | 7e-07     | 0.486      | 2e-09      | Mirrors heating/water costs; captures overall household expenditure                 |
| LOW_ENERGY_FIXED_LIGHT_COUNT                                     | -0.243          | 0.004     | -0.329     | 9e-05      | Moderate negative correlation; efficient lighting can reduce costs                  |
| TOTAL_FLOOR_AREA                                                 | -0.213          | 0.013     | -0.332     | 8e-05      | Negative correlation may reflect economies of scale or atypical dwellings           |
| HEATING_COST_POTENTIAL                                           | 0.182           | 0.034     | 0.267      | 0.0017     | Small-to-moderate; potential costs indicate retrofit opportunities                  |
| HOT_WATER_COST_POTENTIAL                                         | 0.216           | 0.011     | 0.097      | 0.26       | Weak; some signal but not strong                                                    |
| Other variables (SPACE_HEATING_DEMAND, rooms, glazing, lighting) | low correlation |           |            |            | Individually weak predictors                                                        |

**Analysis:**

* **ENERGY_CONSUMPTION_CURRENT** is the dominant continuous predictor of cost intensity.
* **Water and heating costs** also strongly drive total household energy expenditure.
* **Retrofit potential and lighting modernization** are moderate predictors, indicating opportunities for savings.

**Takeaway for archetyping:** Use **ENERGY_CONSUMPTION_CURRENT** as the primary continuous variable. Include **water/heating costs** or **retrofit potential** if subtypes are desired.

---

## **2. Binary Variables**

| Variable                 | PointBiserial_r | p_value | Notes                                                    |
| ------------------------ | --------------- | ------- | -------------------------------------------------------- |
| FLAT_TOP_STOREY          | -0.246          | 0.524   | Weak negative correlation; not statistically significant |
| SOLAR_WATER_HEATING_FLAG | -0.084          | 0.344   | Very weak; negligible for cost intensity                 |

**Analysis:**

* Binary variables have minimal influence on cost intensity.
* Could serve as **descriptive attributes** for archetypes (e.g., noting solar water heating).

**Takeaway for archetyping:** Focus on continuous and categorical variables; binary features are descriptive only.

---

## **3. Categorical Variables (ANOVA / Eta²)**

| Variable                                                                      | F_stat       | p_value | Eta²  | Notes                                                                     |
| ----------------------------------------------------------------------------- | ------------ | ------- | ----- | ------------------------------------------------------------------------- |
| MAIN_HEATING_CONTROLS                                                         | 2.47         | 0.00087 | 0.358 | Strong effect; controls influence cost patterns                           |
| MAIN_FUEL                                                                     | 2.78         | 0.0023  | 0.220 | Significant; fuel type affects energy costs                               |
| MAIN_HEATING_CATEGORY                                                         | 5.64         | 3.3e-05 | 0.214 | Important categorical predictor; heating system type shapes cost profile  |
| TRANSACTION_TYPE                                                              | 3.00         | 0.006   | 0.141 | Moderate effect; occupancy/usage patterns influence household expenditure |
| ENERGY_TARIFF                                                                 | 4.20         | 0.0072  | 0.092 | Small-to-moderate effect; tariff influences cost                          |
| MECHANICAL_VENTILATION                                                        | 4.87         | 0.0091  | 0.069 | Minor effect                                                              |
| TENURE                                                                        | 3.21         | 0.025   | 0.068 | Minor effect                                                              |
| Other variables (CONSTRUCTION_AGE_BAND, PROPERTY_TYPE, BUILT_FORM, DATA_ZONE) | weak/not sig | >0.3    | <0.08 | Negligible effect                                                         |

**Analysis:**

* **Heating system type, controls, and fuel** are the most significant categorical contributors to cost intensity.
* Occupancy, tariff, and minor building features are secondary contributors.

**Takeaway for archetyping:** Use **MAIN_HEATING_CATEGORY, MAIN_HEATING_CONTROLS, MAIN_FUEL** as key categorical axes.

---

## **4. Recommended Variables for Cost Archetyping**

**Primary (strong predictors, statistically significant):**

1. **ENERGY_CONSUMPTION_CURRENT** – continuous, main driver of cost intensity
2. **MAIN_HEATING_CONTROLS** – categorical, controls shape cost pattern
3. **MAIN_HEATING_CATEGORY** – categorical, heating type shapes cost profile
4. **MAIN_FUEL** – categorical, significant contributor

**Secondary (descriptive / moderate predictors):**

* 3_YR_ENERGY_SAVINGS_POTENTIAL
* HOT_WATER_COST_CURRENT
* LOW_ENERGY_FIXED_LIGHT_COUNT
* Binary flags (solar water heating, flat top storey) as descriptive labels

---

## **5. Suggested Procedure for Cost Archetyping**

1. **Step 1 – Select variables:**

   * Continuous: `ENERGY_CONSUMPTION_CURRENT` (+ optional 3_YR_ENERGY_SAVINGS_POTENTIAL)
   * Categorical: `MAIN_HEATING_CATEGORY`, `MAIN_HEATING_CONTROLS`, `MAIN_FUEL`

2. **Step 2 – Initial grouping:**

   * For categorical variables, define archetypes by combinations (e.g., “Gas central heating + basic controls + gas fuel”).
   * For continuous variables, bin into low/medium/high cost intensity to separate archetypes.

3. **Step 3 – Statistical verification:**

   * Check **within-group variance** for continuous variables to ensure groups are homogeneous.
   * Check **between-group variance** to confirm groups are well-separated.

4. **Step 4 – Align with local knowledge:**

   * Compare statistical archetypes with actual Fintry dwellings for interpretability.
   * Adjust bins or categories to match intuitive understanding while retaining statistical separation.

5. **Step 5 – Use archetypes in modeling:**

   * Assign dwellings to cost archetypes.
   * Run PHPP / hourly models per archetype.
   * Store representative cost profiles for future scenario analysis.

---

✅ **Summary:**

* **Most predictive variables:** ENERGY_CONSUMPTION_CURRENT, MAIN_HEATING_CONTROLS, MAIN_HEATING_CATEGORY, MAIN_FUEL
* **Next steps:** Archetypes should be based primarily on energy consumption and heating system characteristics; secondary variables (water cost, lighting efficiency) can refine subtypes.
* **Binary/minor categorical variables** are descriptive only.

---

If you want, I can now **combine PEI, Carbon, and Cost correlations into a single summary table**, so you can see which variables consistently matter across all three metrics — this is very useful for prioritizing your archetype design. Do you want me to do that next?
